In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [2]:
df = pd.read_csv("../data/processed/car_sales_cleaned.csv")

In [3]:
df.head(4)

,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats,brand,car_age
0,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74.00,5.0,Maruti,12
1,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,5.0,Skoda,12
2,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78.00,5.0,Honda,20
3,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,90.00,5.0,Hyundai,16


### Splitting data

In [4]:
X = df.drop(columns=["selling_price"])
y = np.log1p(df["selling_price"])

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Preprocessing | Pipeline

In [8]:
numerical_cols = df.select_dtypes(include="number").columns.to_list()
numerical_cols.remove('selling_price')
categorical_cols = df.select_dtypes(include="object").columns.to_list()
categorical_cols.remove("owner")
print(numerical_cols, categorical_cols)

['km_driven', 'mileage', 'engine', 'max_power', 'seats', 'car_age'] ['fuel', 'seller_type', 'transmission', 'brand']


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical - OHE (fuel, seller_type, transmission, brand)
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Ordinal (owner only)
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['First Owner', 'Second Owner', 
                               'Third Owner', 'Fourth & Above Owner', 
                               'Test Drive Car']]))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols),
    ('ordinal', ordinal_pipeline, ['owner'])
])


# Linear Regression
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

full_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [11]:
y_pred = full_pipeline.predict(X_test)

# reverse log transform
y_pred_actual = np.expm1(y_pred)
y_test_actual = np.expm1(y_test)

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print('R2:', round(r2_score(y_test_actual, y_pred_actual), 4))
print('MAE:', round(mean_absolute_error(y_test_actual, y_pred_actual), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test_actual, y_pred_actual)), 2))

R2: 0.8671
MAE: 91413.11
RMSE: 170751.93


In [21]:
# trying Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor

randomforest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

randomforest_pipeline.fit(X_train, y_train)

y_pred_rf = randomforest_pipeline.predict(X_test)
y_pred_rf_actual = np.expm1(y_pred_rf)

print('R2:', round(r2_score(y_test_actual, y_pred_rf_actual), 4))
print('MAE:', round(mean_absolute_error(y_test_actual, y_pred_rf_actual), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test_actual, y_pred_rf_actual)), 2))

R2: 0.9245
MAE: 73340.23
RMSE: 128706.34


In [22]:
# trying Xgboost
from xgboost import XGBRegressor

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(n_estimators=100, random_state=42))
])

xgb_pipeline.fit(X_train, y_train)

y_pred_xgb = xgb_pipeline.predict(X_test)
y_pred_xgb_actual = np.expm1(y_pred_xgb)

print('R2:', round(r2_score(y_test_actual, y_pred_xgb_actual), 4))
print('MAE:', round(mean_absolute_error(y_test_actual, y_pred_xgb_actual), 2))
print('RMSE:', round(np.sqrt(mean_squared_error(y_test_actual, y_pred_xgb_actual)), 2))

R2: 0.9201
MAE: 72094.74
RMSE: 132338.34


### Cross-validation

In [24]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(randomforest_pipeline, X_train, y_train, cv=5, scoring='r2')

print('CV R2 scores:', cv_scores.round(4))
print('Mean R2:', round(cv_scores.mean(), 4))
print('Std:', round(cv_scores.std(), 4))

CV R2 scores: [0.9074 0.9078 0.9111 0.9063 0.9156]
Mean R2: 0.9097
Std: 0.0034


In [29]:
print("Train R2: ", round(randomforest_pipeline.score(X_train, y_train), 4))
print("Test R2: ", round(randomforest_pipeline.score(X_test, y_test), 4))

Train R2:  0.9868
Test R2:  0.9016


### Saving model with joblib

In [31]:
import joblib

joblib.dump(randomforest_pipeline, '../models/model_v1.pkl')
print('Saved.')

Saved.


In [32]:
# loading back and checking with raw unprocessed data
pipeline = joblib.load("../models/model_v1.pkl")

In [33]:
sample = pd.DataFrame([{
    'km_driven': 45000,
    'fuel': 'Petrol',
    'seller_type': 'Individual',
    'transmission': 'Manual',
    'owner': 'First Owner',
    'mileage': 18.5,
    'engine': 1197.0,
    'max_power': 82.0,
    'seats': 5.0,
    'brand': 'Maruti',
    'car_age': 5
}])
prediction = pipeline.predict(sample)
actual_price = np.expm1(prediction[0])
print(f'Predicted price: ₹{actual_price:,.0f}')

Predicted price: ₹517,823
